# ReAct Agent: Ollama 8B + LangChain/LangGraph + Streamlit + Cloudflare Tunnel
논문 **「ReAct: Synergizing Reasoning and Acting in Language Models (ICLR 2023)」** 이론을 OpenAI API Key 없이 **Colab 로컬 Ollama(Llama 3.1 8B)**로 구현하고, **Streamlit 웹 앱 + Cloudflare Tunnel**로 즉시 시연할 수 있는 통합 노트북입니다.

### 주요 수정 사항
1. **패키지 선설치 및 버전 고정**: `langchain-ollama`, `langgraph`, `ddgs`, `streamlit`을 최상단에서 먼저 설치
2. **Tool Calling 지원 8B 모델 적용**: `llama3` 대신 Tool Calling을 네이티브 지원하는 `llama3.1:8b` 및 `langchain_ollama.ChatOllama` 사용
3. **네임스페이스 충돌 제거**: `create_classic_react_agent` vs `create_langgraph_react_agent` 별칭 분리
4. **Cloudflare Tunnel 웹 시연**: `app.py` 생성 후 `https://*.trycloudflare.com` 퍼블릭 URL 자동 출력

In [ ]:
# 1) Colab Ubuntu 환경에서 Ollama 설치에 필요한 유틸리티 설치
!apt-get -qq update && apt-get -qq install -y zstd pciutils curl

# 2) Colab Python 3.13 사전 설치 패키지(langgraph-prebuilt 1.x 등)와 버전 충돌이 없도록 함께 동기화 업그레이드
!pip install -q --upgrade "langgraph>=1.0.0" "langgraph-prebuilt>=1.0.0" "langchain>=1.0.0" "langchain-classic>=1.0.0" "langchain-community" "langchain-ollama" "langchain-core" "wikipedia" "ddgs" "beautifulsoup4" "streamlit"

import importlib
importlib.invalidate_caches()
print("✅ 모든 Python 패키지 동기화 설치 완료 (이미 이전 셀을 실행했던 세션이라면 [런타임 > 세션 다시 시작] 후 다음 셀부터 실행하세요)")

In [ ]:
import shutil
import subprocess
import time
import requests

# 1) Ollama 설치 (이미 설치되어 있으면 건너뜀)
if shutil.which("ollama") is None:
    print("⏳ Ollama 설치 중...")
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

# 2) 기존 프로세스 정리 후 백그라운드 실행
subprocess.run(["pkill", "-f", "ollama serve"], check=False)
time.sleep(1)
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 3) /api/tags 헬스체크 폴링 (최대 30초 대기)
for _ in range(30):
    try:
        r = requests.get("http://127.0.0.1:11434/api/tags", timeout=1)
        if r.status_code == 200:
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama 서버가 시작되지 않았습니다.")

print("✅ Ollama 서버 정상 가동 확인 (127.0.0.1:11434)")

# 4) Tool Calling을 네이티브 지원하는 Llama 3.1 (8B) 모델 다운로드 및 검증
print("⏳ llama3.1:8b 모델 다운로드 중 (최초 1회 약 4.9GB)...")
subprocess.run(["ollama", "pull", "llama3.1:8b"], check=True)

tags_resp = requests.get("http://127.0.0.1:11434/api/tags", timeout=5)
tags_resp.raise_for_status()
installed = [m["name"] for m in tags_resp.json().get("models", [])]
if not any("llama3.1:8b" in m for m in installed):
    raise RuntimeError(f"llama3.1:8b 다운로드 확인 실패. 현재 모델: {installed}")
print("✅ 준비 완료된 Ollama 모델:", installed)

In [ ]:
import ast
import operator
import re
import shutil
import subprocess
import time
import requests
from ddgs import DDGS
from langchain_ollama import ChatOllama
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool

# [핵심 수정 1] LangChain v1+ (langchain_classic) 및 구버전 (langchain.agents) 모두 호환되도록 Classic ReAct 임포트
try:
    from langchain_classic.agents import create_react_agent as create_classic_react_agent, AgentExecutor
except ImportError:
    from langchain.agents import create_react_agent as create_classic_react_agent, AgentExecutor

# [핵심 수정 2] LangChain v1+ 표준 create_agent 우선 사용, 구버전 환경에서는 langgraph.prebuilt 폴백 (langgraph._internal 충돌 방지)
try:
    from langchain.agents import create_agent as create_modern_agent
except ImportError:
    from langgraph.prebuilt import create_react_agent as create_modern_agent

# [핵심 수정 3] Ollama 서버 & 모델(llama3.1:8b) 자가 복구 체크 (세션 재시작/셀 단독 실행 시 404 방지)
def ensure_ollama_ready(model_name: str = "llama3.1:8b"):
    if shutil.which("ollama") is None:
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    try:
        r = requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        alive = r.status_code == 200
    except Exception:
        alive = False
    if not alive:
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for _ in range(30):
            try:
                r = requests.get("http://127.0.0.1:11434/api/tags", timeout=1)
                if r.status_code == 200:
                    break
            except Exception:
                pass
            time.sleep(1)
        else:
            raise RuntimeError("Ollama 서버를 시작할 수 없습니다.")
    models = [m["name"] for m in requests.get("http://127.0.0.1:11434/api/tags", timeout=5).json().get("models", [])]
    if not any(model_name in m for m in models):
        print(f"⏳ '{model_name}' 모델이 없어 자동 다운로드합니다...")
        subprocess.run(["ollama", "pull", model_name], check=True)

MODEL_NAME = "llama3.1:8b"
ensure_ollama_ready(MODEL_NAME)

# 1) 공식 langchain-ollama ChatOllama (Llama 3.1 8B)
llm = ChatOllama(model=MODEL_NAME, temperature=0)

# 2) 도구 정의 (Wikipedia + 안전한 AST 계산기 + DuckDuckGo)
wikipedia = WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=700)
wiki_tool = WikipediaQueryRun(api_wrapper=wikipedia)

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}
def _safe_eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("Unsupported expression")

@tool
def safe_calculator(expression: str) -> str:
    """Evaluate a pure mathematical expression safely (e.g., '2884 + 220')."""
    try:
        return str(_safe_eval(ast.parse(expression.strip(), mode="eval").body))
    except Exception as e:
        return f"Calculator Error: {e}"

from concurrent.futures import ThreadPoolExecutor, as_completed
import urllib.parse
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup

@tool
def web_search(query: str) -> str:
    """Search DuckDuckGo, Google, Naver, and Wikipedia in parallel for verified facts."""
    q = query.strip()
    is_ko = bool(re.search(r"[가-힣]", q))
    region = "kr-kr" if is_ko else "wt-wt"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/124.0.0.0 Safari/537.36"}

    def _ddg():
        try:
            try:
                hits = list(DDGS().text(q, region=region, max_results=2, backend="duckduckgo"))
            except TypeError:
                hits = list(DDGS().text(q, max_results=2))
            return [f"[DuckDuckGo] {r.get('title','')}: {str(r.get('body',''))[:150]}" for r in hits]
        except Exception:
            return []

    def _google():
        res = []
        for bk in ("google", "bing", "brave"):
            try:
                hits = list(DDGS().text(q, region=region, max_results=2, backend=bk))
                res = [f"[Google/{bk}] {r.get('title','')}: {str(r.get('body',''))[:150]}" for r in hits]
                if res:
                    break
            except Exception:
                continue
        if not res:
            try:
                hl, gl, ceid = ("ko", "KR", "KR:ko") if is_ko else ("en-US", "US", "US:en")
                rss = requests.get(f"https://news.google.com/rss/search?q={urllib.parse.quote_plus(q)}&hl={hl}&gl={gl}&ceid={ceid}", headers=headers, timeout=5)
                if rss.status_code == 200:
                    root = ET.fromstring(rss.text)
                    res = [f"[Google News] {it.find('title').text.strip()}" for it in root.findall(".//item")[:2] if it.find("title") is not None and it.find("title").text]
            except Exception:
                pass
        return res

    def _naver():
        res = []
        try:
            r = requests.get(f"https://search.naver.com/search.naver?where=nexearch&query={urllib.parse.quote_plus(q)}", headers=headers, timeout=5)
            if r.status_code == 200:
                soup = BeautifulSoup(r.text, "html.parser")
                for el in soup.select(".total_dsc, .news_dsc, .api_txt_lines.dsc_txt")[:2]:
                    txt = el.get_text(" ", strip=True)[:160]
                    if txt:
                        res.append(f"[Naver] {txt}")
        except Exception:
            pass
        return res

    merged = []
    with ThreadPoolExecutor(max_workers=3) as ex:
        for f in as_completed([ex.submit(_ddg), ex.submit(_google), ex.submit(_naver)]):
            merged.extend(f.result())
    return " | ".join(merged) or "No results."

# 3) Classic ReAct 실행 (LangChain v1+에서도 안전하도록 hub 임포트를 try 내부에 배치 + 프롬프트 폴백 내장)
try:
    from langchain import hub
    react_prompt = hub.pull("hwchase17/react")
except Exception:
    react_prompt = PromptTemplate.from_template(
        "Answer the following questions as best you can. You have access to the following tools:\n\n"
        "{tools}\n\nUse the following format:\n"
        "Question: {input}\nThought: you should always think about what to do\n"
        "Action: the action to take, should be one of [{tool_names}]\n"
        "Action Input: the input to the action\nObservation: the result of the action\n"
        "... (repeat Thought/Action/Action Input/Observation)\n"
        "Thought: I now know the final answer\nFinal Answer: the final answer\n\n"
        "Begin!\n\nQuestion: {input}\nThought:{agent_scratchpad}"
    )

classic_tools = [wiki_tool, safe_calculator]
classic_agent = create_classic_react_agent(llm=llm, tools=classic_tools, prompt=react_prompt)
classic_executor = AgentExecutor(
    agent=classic_agent,
    tools=classic_tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=8,
)

q = "What is the elevation range for the area that the eastern sector of the Colorado orogeny extends into?"
print("▶ Classic ReAct 테스트:")
print(classic_executor.invoke({"input": q})["output"])

# 4) LangGraph Tool-Calling ReAct 실행 (이름 충돌 없이 정상 동작)
@tool
def search_wikipedia(query: str) -> str:
    """Search Wikipedia and return a concise summary."""
    return wikipedia.run(query)

graph_agent = create_modern_agent(model=llm, tools=[search_wikipedia, web_search, safe_calculator])
print("\n▶ LangGraph ReAct 테스트:")
graph_res = graph_agent.invoke({"messages": [HumanMessage(content=q)]})
print(graph_res["messages"][-1].content)

## Streamlit 웹 앱 + Cloudflare Tunnel 배포
아래 셀은 **ReAct 논문 기반 Streamlit 웹 앱(`app.py`)**을 생성하고, 별도의 계정이나 토큰 없이 **`cloudflared` Quick Tunnel**을 통해 누구나 접속 가능한 실시간 웹 시연 URL(`https://*.trycloudflare.com`)을 발급합니다.

In [ ]:
%%writefile app.py
import ast
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import operator
import re
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
import requests
import streamlit as st
import wikipedia
from ddgs import DDGS
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
try:
    from langchain.agents import create_agent as create_modern_agent
except ImportError:
    from langgraph.prebuilt import create_react_agent as create_modern_agent

st.set_page_config(
    page_title="ReAct (ICLR 2023) + RAG · Ollama 8B Live Lab",
    page_icon="⚡",
    layout="wide",
)

# ---------------------------------------------------------
# Simple In-Memory RAG System for Streamlit
# ---------------------------------------------------------
def chunk_text(text, chunk_size=500, overlap=100):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks

def simple_rag_search(query: str, chunks: list, top_k: int = 3) -> str:
    if not chunks:
        return "사용자가 업로드한 문서가 없거나 비어 있습니다."

    # BM25 또는 간단한 TF-IDF 기반 단어 매칭 검색 구현
    query_words = set(re.findall(r"[가-힣a-zA-Z0-9]+", query.lower()))
    scored_chunks = []
    for idx, chunk in enumerate(chunks):
        chunk_words = re.findall(r"[가-힣a-zA-Z0-9]+", chunk.lower())
        match_count = sum(1 for w in query_words if w in chunk_words)
        score = match_count / (len(query_words) + 1)
        scored_chunks.append((score, idx, chunk))

    scored_chunks.sort(key=lambda x: (-x[0], x[1]))
    results = []
    for score, idx, chunk in scored_chunks[:top_k]:
        if score > 0:
            results.append(f"[업로드 문서 발췌 {idx+1}] {chunk[:250]}...")

    if not results:
        return "업로드된 문서에서 질문과 관련된 내용을 찾을 수 없습니다."
    return "\n\n".join(results)

# ---------------------------------------------------------
# 1. Safe Math Evaluator & Paper Wikipedia API Environment
# ---------------------------------------------------------
_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
}

def _safe_eval_node(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval_node(node.left), _safe_eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval_node(node.operand))
    raise ValueError("Unsupported expression")

def _detect_wiki_lang(text: str, default_lang: str = "en") -> str:
    return "ko" if (default_lang == "ko" or re.search(r"[가-힣]", text)) else "en"

def wiki_search_api(entity: str, lang: str = "en") -> tuple[str, list[str]]:
    clean = entity.strip().strip("\"'")
    effective_lang = _detect_wiki_lang(clean, lang)
    try:
        wikipedia.set_lang(effective_lang)
        page = wikipedia.page(clean, auto_suggest=False)
        sentences = [s.strip() for s in re.split(r"(?<=[.!?다])\s+", page.content) if s.strip()]
        obs = " ".join(sentences[:6]) if sentences else page.summary[:700]
        return f"[{page.title} ({effective_lang}.wiki)] {obs}", sentences
    except wikipedia.DisambiguationError as e:
        opts = e.options[:6]
        return f"Disambiguation for [{clean}] ({effective_lang}.wiki). Try one of: {opts}", []
    except wikipedia.PageError:
        results = wikipedia.search(clean, results=5)
        return (
            f"Could not find exact [{clean}] ({effective_lang}.wiki). "
            f"Similar: {results}. Try a similar title OR call web_search.",
            [],
        )
    except Exception as exc:
        return f"Wikipedia lookup error: {exc}", []

def wiki_lookup_api(keyword: str, sentences: list[str]) -> str:
    if not sentences:
        return "No Wikipedia page loaded yet. Call search[entity] first."
    key = keyword.strip().strip("\"'").lower()
    matches = [s for s in sentences if key in s.lower()]
    if not matches:
        return f"No sentence containing '{keyword}' in current page."
    return f"(Result 1 / {len(matches)}) {matches[0]}"

_SEARCH_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
}

def _search_duckduckgo(clean_q: str, max_results: int = 3) -> list[str]:
    out: list[str] = []
    region = "kr-kr" if re.search(r"[가-힣]", clean_q) else "wt-wt"
    try:
        try:
            results = list(DDGS().text(clean_q, region=region, max_results=max_results, backend="duckduckgo"))
        except TypeError:
            results = list(DDGS().text(clean_q, max_results=max_results))
        for r in results:
            title = str(r.get("title", "")).strip()
            body = str(r.get("body", "")).strip()[:180]
            if title or body:
                out.append(f"[DuckDuckGo] {title}: {body}")
    except Exception:
        pass
    return out

def _search_google(clean_q: str, max_results: int = 3) -> list[str]:
    out: list[str] = []
    is_ko = bool(re.search(r"[가-힣]", clean_q))
    region = "kr-kr" if is_ko else "wt-wt"
    for backend_name in ("google", "bing", "brave"):
        try:
            results = list(DDGS().text(clean_q, region=region, max_results=max_results, backend=backend_name))
            for r in results:
                title = str(r.get("title", "")).strip()
                body = str(r.get("body", "")).strip()[:180]
                if title or body:
                    tag = "Google" if backend_name == "google" else f"Google/{backend_name.title()}"
                    out.append(f"[{tag}] {title}: {body}")
            if out:
                break
        except Exception:
            continue

    try:
        hl, gl, ceid = ("ko", "KR", "KR:ko") if is_ko else ("en-US", "US", "US:en")
        rss_url = f"https://news.google.com/rss/search?q={urllib.parse.quote_plus(clean_q)}&hl={hl}&gl={gl}&ceid={ceid}"
        resp = requests.get(rss_url, headers=_SEARCH_HEADERS, timeout=6)
        if resp.status_code == 200:
            root = ET.fromstring(resp.text)
            for item in root.findall(".//item")[:2]:
                title_el = item.find("title")
                if title_el is not None and title_el.text:
                    out.append(f"[Google News] {title_el.text.strip()}")
    except Exception:
        pass
    return out[:max_results + 1]

def _search_naver(clean_q: str, max_results: int = 3) -> list[str]:
    out: list[str] = []
    encoded = urllib.parse.quote_plus(clean_q)
    try:
        r_encyc = requests.get(
            f"https://terms.naver.com/search.naver?query={encoded}",
            headers=_SEARCH_HEADERS,
            timeout=6,
        )
        if r_encyc.status_code == 200:
            soup = BeautifulSoup(r_encyc.text, "html.parser")
            for li in soup.select("ul.content_list li")[:2]:
                t_el = li.select_one(".title a") or li.select_one(".title")
                d_el = li.select_one("p.desc") or li.select_one(".desc")
                title = t_el.get_text(" ", strip=True) if t_el else ""
                desc = d_el.get_text(" ", strip=True)[:180] if d_el else ""
                if title or desc:
                    out.append(f"[Naver 지식백과] {title}: {desc}")
    except Exception:
        pass

    try:
        r_web = requests.get(
            f"https://search.naver.com/search.naver?where=nexearch&query={encoded}",
            headers=_SEARCH_HEADERS,
            timeout=6,
        )
        if r_web.status_code == 200:
            soup = BeautifulSoup(r_web.text, "html.parser")
            seen_txt: set[str] = set()
            selectors = [
                (".total_tit", ".total_dsc"),
                (".news_tit", ".news_dsc"),
                ("a.link_tit", ".api_txt_lines"),
                (".question_text", ".answer_text"),
            ]
            for t_sel, d_sel in selectors:
                titles = soup.select(t_sel)
                descs = soup.select(d_sel)
                for idx, t_el in enumerate(titles[:max_results]):
                    t_txt = t_el.get_text(" ", strip=True)
                    d_txt = descs[idx].get_text(" ", strip=True)[:180] if idx < len(descs) else ""
                    combined = f"{t_txt}: {d_txt}".strip(": ")
                    if combined and combined not in seen_txt:
                        seen_txt.add(combined)
                        out.append(f"[Naver] {combined}")
                    if len(out) >= max_results + 2:
                        break
                if len(out) >= max_results + 2:
                    break
            if len(out) < 2:
                for p in soup.select(".api_txt_lines.dsc_txt, .sds-comps-text-type-body1")[:3]:
                    txt = p.get_text(" ", strip=True)[:200]
                    if txt and txt not in seen_txt:
                        seen_txt.add(txt)
                        out.append(f"[Naver] {txt}")
    except Exception:
        pass
    return out[:max_results + 1]

def _search_wiki_fulltext(clean_q: str, max_results: int = 3) -> list[str]:
    out: list[str] = []
    try:
        wiki_lang = "ko" if re.search(r"[가-힣]", clean_q) else "en"
        resp = requests.get(
            f"https://{wiki_lang}.wikipedia.org/w/api.php",
            params={
                "action": "query",
                "list": "search",
                "srsearch": clean_q,
                "srlimit": max_results,
                "format": "json",
            },
            headers={"User-Agent": "ReActColabLab/1.0"},
            timeout=6,
        )
        if resp.status_code == 200:
            hits = resp.json().get("query", {}).get("search", [])
            for h in hits:
                clean_snip = re.sub(r"<[^>]+>", "", h.get("snippet", "")).strip()
                out.append(f"[WikiFullText:{h.get('title','')}] {clean_snip}")
    except Exception:
        pass
    return out

def web_search_api(query: str, engines: list[str] | None = None, max_per_engine: int = 3) -> str:
    clean_q = query.strip()
    active = engines if engines else ["DuckDuckGo", "Google", "Naver", "Wikipedia 본문"]
    engine_map = {
        "DuckDuckGo": _search_duckduckgo,
        "Google": _search_google,
        "Naver": _search_naver,
        "Wikipedia 본문": _search_wiki_fulltext,
    }
    ordered_results: dict[str, list[str]] = {}
    with ThreadPoolExecutor(max_workers=4) as executor:
        future_to_name = {
            executor.submit(engine_map[name], clean_q, max_per_engine): name
            for name in active
            if name in engine_map
        }
        for future in as_completed(future_to_name):
            name = future_to_name[future]
            try:
                ordered_results[name] = future.result()
            except Exception:
                ordered_results[name] = []

    merged: list[str] = []
    seen: set[str] = set()
    for name in active:
        for item in ordered_results.get(name, []):
            norm = item.lower().strip()
            if norm and norm not in seen:
                seen.add(norm)
                merged.append(item)

    return " | ".join(merged) if merged else "No web results found."

def check_ollama_models(base_url: str = "http://127.0.0.1:11434") -> list[str]:
    try:
        r = requests.get(f"{base_url}/api/tags", timeout=3)
        if r.status_code == 200:
            return [m["name"] for m in r.json().get("models", [])]
    except Exception:
        pass
    return []

def ensure_ollama_model(model_name: str, base_url: str = "http://127.0.0.1:11434") -> None:
    models = check_ollama_models(base_url)
    if not any(m == model_name or m.startswith(f"{model_name}:") for m in models):
        with st.spinner(f"Ollama 모델 '{model_name}' 다운로드 중 (최초 1회)..."):
            r = requests.post(f"{base_url}/api/pull", json={"name": model_name, "stream": False}, timeout=900)
            r.raise_for_status()

# ---------------------------------------------------------
# 2. Classic Paper ReAct Loop (Thought -> Action -> Obs)
# ---------------------------------------------------------
def run_react_loop(
    llm: ChatOllama,
    question: str,
    lang: str = "en",
    max_steps: int = 7,
    enable_web: bool = False,
    engines: list[str] | None = None,
    preset_history: list[dict] | None = None,
    rag_chunks: list | None = None,
):
    steps = list(preset_history) if preset_history else []
    page_sentences: list[str] = []

    # Replay prior actions to restore Wikipedia page buffer for HITL resume
    has_verified_obs = False
    for prev in steps:
        act = prev.get("action", "")
        m = re.match(r"^(search|lookup|web_search|rag_retriever|calc|finish)\[(.*)\]$", act, re.I | re.S)
        if m:
            if m.group(1).lower() == "search":
                _, new_sents = wiki_search_api(m.group(2), lang=lang)
                if new_sents:
                    page_sentences = new_sents
                    has_verified_obs = True
            elif m.group(1).lower() in ("web_search", "rag_retriever") and "No results" not in prev.get("observation", ""):
                has_verified_obs = True

    web_rule = (
        '- "web_search": Parallel multi-engine internet search (DuckDuckGo + Google + Naver + Wiki). (action_arg = query)'
        if enable_web
        else ""
    )
    rag_rule = (
        '- "rag_retriever": Search inside the uploaded local document chunks. Always check here first if user refers to upload files. (action_arg = core keyword or phrase)'
        if rag_chunks
        else ""
    )
    target_lang = "Korean" if _detect_wiki_lang(question, lang) == "ko" else "English"
    sys_prompt = f"""You are a strictly grounded ReAct agent implementing Yao et al. (ICLR 2023).
At each step, output strictly valid JSON with keys: "thought", "action_type", "action_arg".
Available action_type values:
{rag_rule}
- "search": Search Wikipedia for an exact entity name from the question (action_arg = entity title in {target_lang}).
- "lookup": Ctrl+F keyword lookup in the currently loaded Wikipedia page (action_arg = keyword).
{web_rule}
- "calc": Evaluate a pure arithmetic expression like "2884 + 220" (action_arg = math expression).
- "finish": Provide the concise final answer ONLY AFTER verifying it from Observations (action_arg = final answer).

CRITICAL ANTI-HALLUCINATION RULES:
1. NEVER guess an unverified name or fact from memory! Check your local document RAG index or search Wikipedia first.
2. Respond ONLY with valid JSON."""

    has_pending_edit = len(steps) > 0 and steps[-1].get("thought") and not steps[-1].get("action")
    start_idx = len(steps) if has_pending_edit else len(steps) + 1

    for step_num in range(start_idx, max_steps + 1):
        is_edited = (step_num == start_idx) and has_pending_edit
        hist_str = "\n".join(
            f"Thought {s['step']}: {s.get('thought','')}\n"
            + (f"Action {s['step']}: {s['action']}\nObservation {s['step']}: {s['observation']}" if s.get("action") else "")
            for s in steps
        )

        if is_edited:
            user_msg = (
                f"Question: {question}\n\nHistory:\n{hist_str}\n\n"
                f"Follow the human-edited Thought {step_num} strictly and output JSON for the next action_type & action_arg."
            )
        else:
            user_msg = f"Question: {question}\n\nHistory:\n{hist_str or '(Start)'}\n\nGenerate Step {step_num} JSON."

        raw = llm.invoke([SystemMessage(content=sys_prompt), HumanMessage(content=user_msg)]).content
        json_match = re.search(r"\{.*\}", raw, re.S)
        try:
            parsed = json.loads(json_match.group(0) if json_match else raw)
        except Exception:
            parsed = {"thought": raw.strip()[:200], "action_type": "finish", "action_arg": raw.strip()}

        thought = steps[-1]["thought"] if is_edited else str(parsed.get("thought", ""))
        act_type = str(parsed.get("action_type", "search")).lower().strip()
        act_arg = str(parsed.get("action_arg", "")).strip()

        act_str = f"{act_type}[{act_arg}]"

        if act_type == "rag_retriever" and rag_chunks:
            obs = simple_rag_search(act_arg, rag_chunks)
            has_verified_obs = True
        elif act_type == "search":
            obs, new_sents = wiki_search_api(act_arg, lang=lang)
            if new_sents:
                page_sentences = new_sents
                has_verified_obs = True
        elif act_type == "lookup":
            obs = wiki_lookup_api(act_arg, page_sentences)
        elif act_type == "web_search" and enable_web:
            obs = web_search_api(act_arg, engines=engines)
            if "No web results" not in obs:
                has_verified_obs = True
        elif act_type == "calc":
            try:
                val = _safe_eval_node(ast.parse(act_arg, mode="eval").body)
                obs = f"Result = {val}"
                has_verified_obs = True
            except Exception as e:
                obs = f"Calc error: {e}"
        else:
            act_str = f"finish[{act_arg}]"
            obs = "Episode finished."

        if is_edited:
            steps[-1]["action"] = act_str
            steps[-1]["observation"] = obs
        else:
            steps.append({"step": step_num, "thought": thought, "action": act_str, "observation": obs})

        if act_type == "finish":
            return steps, act_arg

    return steps, "최대 스텝 내에 검증된 확정 답변을 생성하지 못했습니다."

# ---------------------------------------------------------
# 3. Streamlit UI
# ---------------------------------------------------------
st.title("ReAct + Simple RAG: Synergizing Reasoning, Acting & Documents")
st.caption("로컬 업로드 문서(RAG) + 실시간 다중 인터넷 검색이 유기적으로 통합된 Ollama Llama 3.1 8B 데모")

available_models = check_ollama_models()
with st.sidebar:
    st.header("1. Ollama 8B & 에이전트 설정")
    if available_models:
        st.success(f"Ollama 연결됨 ({len(available_models)}개 모델)")
        default_idx = next((i for i, m in enumerate(available_models) if "llama3.1" in m), 0)
        model_name = st.selectbox("모델 선택", available_models, index=default_idx)
    else:
        st.error("Ollama 서버(127.0.0.1:11434)에 연결할 수 없습니다.")
        model_name = st.text_input("모델명", value="llama3.1:8b")

    mode = st.radio(
        "추론·행동 모드",
        ["ReAct (Thought → Action → Obs)", "CoT (Reason Only · 비교군)", "LangGraph Tool-Calling Agent"],
    )
    wiki_lang = st.selectbox("Wikipedia 언어", ["en", "ko"], index=0)
    max_steps = st.slider("최대 추론 스텝", 3, 10, 7)

    st.header("2. RAG 지식베이스 설정")
    uploaded_file = st.file_uploader("로컬 텍스트 문서 업로드 (*.txt)", type=["txt"])
    rag_chunks = []
    if uploaded_file is not None:
        file_contents = uploaded_file.read().decode("utf-8")
        rag_chunks = chunk_text(file_contents)
        st.success(f"문서 처리 완료! 총 {len(rag_chunks)}개의 청크 분할됨.")

    st.header("3. 외부 검색망 설정")
    enable_web = st.checkbox("병렬 멀티 검색엔진 활성화", value=True)
    selected_engines = st.multiselect(
        "병렬 검색 엔진 선택",
        ["DuckDuckGo", "Google", "Naver", "Wikipedia 본문"],
        default=["DuckDuckGo", "Google", "Naver", "Wikipedia 본문"],
        disabled=not enable_web,
    )

# Layout 분할: 메인 에이전트 영역과 실시간 대화창 영역을 탭으로 분할하여 시각성 향상
tab_agent, tab_chat = st.tabs(["🤖 ReAct/RAG 에이전트 분석기", "💬 OLLAMA 8B 실시간 대화창"])

with tab_agent:
    PRESETS = {
        "RAG 전용 테스트 (문서 업로드 후 질문)": "업로드된 문서의 핵심 내용을 바탕으로 핵심 결론을 도출해줘.",
        "HotpotQA · Colorado orogeny (논문 Prompt 예시)": "What is the elevation range for the area that the eastern sector of the Colorado orogeny extends into?",
        "한국어 다중 홉 · 파친코 원작 작가": "드라마 파친코의 원작 소설을 쓴 작가는 어느 나라에서 태어났는가?",
    }

    preset_choice = st.selectbox("논문 벤치마크 프리셋 질문", ["직접 입력"] + list(PRESETS.keys()))
    default_q = PRESETS.get(preset_choice, "")
    question = st.text_area("질문 입력", value=default_q, height=80)

    if "react_steps" not in st.session_state:
        st.session_state.react_steps = []
        st.session_state.react_answer = ""
        st.session_state.last_question = ""

    if st.button("에이전트 실행", type="primary"):
        ensure_ollama_model(model_name)
        llm = ChatOllama(model=model_name, temperature=0)
        st.session_state.last_question = question

        if mode.startswith("CoT"):
            with st.spinner("CoT (Reason Only) 실행 중..."):
                resp = llm.invoke(f"Question: {question}\nLet's think step by step using only internal knowledge.").content
            st.session_state.react_steps = [{"step": 1, "thought": resp, "action": "finish[CoT]", "observation": "외부 검색 미사용"}]
            st.session_state.react_answer = resp
        elif mode.startswith("LangGraph"):
            with st.spinner("LangGraph ReAct Agent (RAG + 인터넷 검색) 실행 중..."):
                lg_sentences: list[str] = []

                @tool
                def rag_retriever(query: str) -> str:
                    """Search and retrieve matching content from locally uploaded text documents."""
                    return simple_rag_search(query, rag_chunks)

                @tool
                def search_wiki(query: str) -> str:
                    """Search Wikipedia for a core topic or entity."""
                    obs, new_sents = wiki_search_api(query, lang=wiki_lang)
                    if new_sents:
                        lg_sentences.clear()
                        lg_sentences.extend(new_sents)
                    return obs

                @tool
                def web_search(query: str) -> str:
                    """Search the live internet in parallel. ALWAYS call this when Wikipedia or local RAG misses exact facts."""
                    return web_search_api(query, engines=selected_engines)

                lg_tools = [rag_retriever, search_wiki, web_search]
                lg_sys = (
                    "You are a strictly grounded ReAct research agent.\n"
                    "Always search local RAG documents using rag_retriever first if the query is about user uploaded files. "
                    "Never hallucinate answers.\n"
                    "Respond in Korean."
                )
                agent = create_modern_agent(model=llm, tools=lg_tools)
                res = agent.invoke({"messages": [SystemMessage(content=lg_sys), HumanMessage(content=question)]})

                trace = []
                for idx, msg in enumerate(res["messages"], start=1):
                    if msg.type == "system":
                        continue
                    trace.append({
                        "step": len(trace) + 1,
                        "thought": f"[{msg.type.upper()}]",
                        "action": str(getattr(msg, "tool_calls", "")) or getattr(msg, "name", "") or msg.type,
                        "observation": str(msg.content)[:700],
                    })
                st.session_state.react_steps = trace
                st.session_state.react_answer = str(res["messages"][-1].content)
        else:
            with st.spinner("ReAct 병렬 검색 루프 실행 중..."):
                steps, ans = run_react_loop(
                    llm, question, lang=wiki_lang, max_steps=max_steps,
                    enable_web=enable_web, engines=selected_engines, rag_chunks=rag_chunks
                )
                st.session_state.react_steps = steps
                st.session_state.react_answer = ans

    if st.session_state.react_steps:
        st.subheader("최종 답변")
        st.info(st.session_state.react_answer)

        st.subheader("Thought · Action · Observation 실행 궤적 & HITL 사고 편집")
        for idx, s in enumerate(st.session_state.react_steps):
            with st.expander(f"Step {s['step']} — {s.get('action', '')}", expanded=True):
                if s.get("thought"):
                    st.markdown(f"**Thought {s['step']}:** {s['thought']}")
                st.code(f"Action {s['step']}: {s.get('action', '')}\nObservation {s['step']}: {s.get('observation', '')}")

with tab_chat:
    st.subheader("💬 OLLAMA 8B와 자유롭게 나누는 다이렉트 챗")
    st.caption("에이전트 제약 없이 로컬 LLM 모델과 자유 대화를 나누거나, 업로드된 문서 내용을 참고하도록 할 수 있습니다.")

    if "chat_history" not in st.session_state:
        st.session_state.chat_history = []

    # 챗 기록 화면 출력
    for msg in st.session_state.chat_history:
        with st.chat_message(msg["role"]):
            st.write(msg["content"])

    # 입력 필드 및 처리
    if user_prompt := st.chat_input("Ollama 8B 모델에게 무엇이든 물어보세요..."):
        # 사용자 메시지 저장 및 표시
        st.session_state.chat_history.append({"role": "user", "content": user_prompt})
        with st.chat_message("user"):
            st.write(user_prompt)

        # 답변 생성
        with st.chat_message("assistant"):
            with st.spinner("응답 생성 및 최신 검색 중..."):
                try:
                    ensure_ollama_model(model_name)
                    raw_chat_llm = ChatOllama(model=model_name, temperature=0.3)

                    # [개선 핵심] 다이렉트 챗에서도 심각한 환각을 해결하기 위해 실시간 다중 인터넷 엔진 검색을 무조건 가동
                    web_search_context = web_search_api(user_prompt, engines=selected_engines, max_per_engine=2)

                    # RAG 문서 검색 컨텍스트 연동
                    rag_context = simple_rag_search(user_prompt, rag_chunks, top_k=3) if rag_chunks else ""

                    chat_messages = []
                    system_context_msg = (
                        "당신은 사용자와 자유롭게 대화하는 성실한 AI 어시스턴트입니다.\n\n"
                        "[중요 지침]\n"
                        "- 최신 정보 및 모르는 역사적/인물적 사실에 대해서는 아래 제공된 실시간 검색 엔진 컨텍스트를 전적으로 신뢰하여 거짓 없이 사실만을 대답하세요.\n"
                        "- 사용자의 입력에 상응하는 최신 지식을 최우선으로 반영하고, 절대 지어낸 정보(예: 이민진 작가의 소설을 임의의 예능 프로그램명으로 답변하는 등)를 포함하지 마세요.\n\n"
                        f"[실시간 인터넷 검색 결과]:\n{web_search_context}\n\n"
                    )
                    if rag_context:
                        system_context_msg += f"[사용자 업로드 로컬 문서 발췌]:\n{rag_context}\n\n"

                    chat_messages.append(SystemMessage(content=system_context_msg))

                    # 과거 대화 메모리 전달
                    for h in st.session_state.chat_history[-6:]:
                        if h["role"] == "user":
                            chat_messages.append(HumanMessage(content=h["content"]))
                        else:
                            chat_messages.append(HumanMessage(content=f"AI 이전 대답: {h['content']}")) # 피드백 루프

                    response = raw_chat_llm.invoke(chat_messages).content
                    st.write(response)
                    st.session_state.chat_history.append({"role": "assistant", "content": response})
                except Exception as e:
                    st.error(f"Ollama 호출 오류: {e}")


In [ ]:
import re
import time

# 배포 후 터널 안정화를 위해 잠시 대기 후 URL 다시 추출
time.sleep(2)

with open("cloudflared.log", "r") as f:
    log_data = f.read()

urls = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_data)
if urls:
    print(f"🚀 최종 확인된 Streamlit 접속 주소: {urls[-1]}")
else:
    print("⚠️ 아직 터널 주소가 완전히 할당되지 않았습니다. 로그 마지막 부분:\n", log_data[-500:])